# LeetCode 1589 — Maximum Sum Obtained of Any Permutation bằng Python

Link đề: https://leetcode.com/problems/maximum-sum-obtained-of-any-permutation/ · Độ khó: **Medium** · Chủ đề: Array, Greedy, Sorting, Prefix Sum

Skeleton LeetCode:

```python
class Solution:
    def maxSumRangeQuery(self, nums: List[int], requests: List[List[int]]) -> int:
        ...
```

## 1. Mục tiêu học

1. Đổi góc nhìn: tổng của mọi truy vấn = `Σ nums[i] · freq[i]`, với `freq[i]` = **số truy vấn phủ chỉ số `i`** — vị trí quan trọng là "được đếm bao nhiêu lần".
2. Viết baseline **thử mọi hoán vị** (`n` nhỏ) để có chuẩn đối chiếu.
3. Tính `freq` hiệu quả bằng **mảng hiệu (difference array)**: `+1` tại `start`, `−1` tại `end + 1`, rồi cộng dồn tiền tố.
4. Áp dụng **bất đẳng thức hoán vị (rearrangement inequality)**: ghép số lớn nhất với tần suất lớn nhất ⇒ tổng lớn nhất.
5. Lấy **modulo `10⁹ + 7`** đúng chỗ và biết dùng **sắp xếp đếm** khi giá trị nhỏ.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums` và mảng truy vấn `requests`, trong đó `requests[i] = [startᵢ, endᵢ]`. Truy vấn thứ `i` hỏi tổng `nums[startᵢ] + nums[startᵢ + 1] + … + nums[endᵢ]` (chỉ số từ 0, hai đầu bao gồm).

Trả về **tổng lớn nhất** của tất cả các truy vấn, xét trên **mọi hoán vị** của `nums`. Kết quả lấy **modulo `10⁹ + 7`**.

**Ví dụ 1:** `nums = [1,2,3,4,5]`, `requests = [[1,3],[0,1]]` → `19`. Hoán vị `[3,5,4,2,1]`: truy vấn 0 → `5+4+2 = 11`, truy vấn 1 → `3+5 = 8`, tổng `19`.

**Ví dụ 2:** `nums = [1,2,3,4,5,6]`, `requests = [[0,1]]` → `11`. Hoán vị `[6,5,4,3,2,1]`.

**Ví dụ 3:** `nums = [1,2,3,4,5,10]`, `requests = [[0,2],[1,3],[1,1]]` → `47`. Hoán vị `[4,10,5,3,2,1]` cho các tổng `[19,18,10]`.

**Ràng buộc:**
- `n == nums.length`, `1 ≤ n ≤ 10⁵`
- `0 ≤ nums[i] ≤ 10⁵`
- `1 ≤ requests.length ≤ 10⁵`, `requests[i].length == 2`
- `0 ≤ startᵢ ≤ endᵢ < n`

In [1]:
import random
import time
from itertools import accumulate, permutations
from typing import List

MOD = 10**9 + 7
examples = [
    ([1, 2, 3, 4, 5], [[1, 3], [0, 1]], 19),
    ([1, 2, 3, 4, 5, 6], [[0, 1]], 11),
    ([1, 2, 3, 4, 5, 10], [[0, 2], [1, 3], [1, 1]], 47),
]
for nums, req, want in examples:
    print(f"nums={nums}, requests={req} -> mong đợi {want}")

nums=[1, 2, 3, 4, 5], requests=[[1, 3], [0, 1]] -> mong đợi 19
nums=[1, 2, 3, 4, 5, 6], requests=[[0, 1]] -> mong đợi 11
nums=[1, 2, 3, 4, 5, 10], requests=[[0, 2], [1, 3], [1, 1]] -> mong đợi 47


## 3. Kiến thức nền

### 3.1 Tổng đoạn bằng mảng tiền tố

`pre[k] = nums[0] + … + nums[k-1]`; tổng `nums[l..r] = pre[r+1] − pre[l]`. Dùng để tính nhanh tổng các truy vấn cho một hoán vị cố định.

In [2]:
arr = [3, 5, 4, 2, 1]
pre = [0] + list(accumulate(arr))
print("pre =", pre)
for l, r in [[1, 3], [0, 1]]:
    print(f"tổng arr[{l}..{r}] =", pre[r + 1] - pre[l])

pre = [0, 3, 8, 12, 14, 15]
tổng arr[1..3] = 11
tổng arr[0..1] = 8


### 3.2 Mảng hiệu (difference array)

Muốn cộng `+1` cho cả đoạn `[l, r]` nhiều lần: thay vì lặp từng phần tử, ghi `d[l] += 1`, `d[r+1] −= 1`; cuối cùng cộng dồn tiền tố `d` ⇒ giá trị tại mỗi vị trí. Mỗi cập nhật `O(1)`, cộng dồn một lần `O(n)`. Mảng `d` cần **`n + 1`** phần tử vì `r + 1` có thể bằng `n`.

In [3]:
n = 6
d = [0] * (n + 1)
for l, r in [[0, 2], [1, 3], [1, 1]]:
    d[l] += 1
    d[r + 1] -= 1
print("mảng hiệu d =", d)
freq = list(accumulate(d))[:n]
print("freq (cộng dồn) =", freq)

mảng hiệu d = [1, 2, -1, -1, -1, 0, 0]
freq (cộng dồn) = [1, 3, 2, 1, 0, 0]


### 3.3 Bất đẳng thức hoán vị (rearrangement inequality)

Với hai dãy số thực `a` và `b`, tổng `Σ a[i] · b[σ(i)]` **lớn nhất** khi cả hai được sắp **cùng chiều** (lớn ghép lớn) và **nhỏ nhất** khi sắp **ngược chiều**.

In [4]:
a, b = [1, 2, 3], [10, 20, 30]
for p in permutations(b):
    print(p, "->", sum(x * y for x, y in zip(a, p)))
print("max khi cùng chiều:", sum(x * y for x, y in zip(sorted(a), sorted(b))))

(10, 20, 30) -> 140
(10, 30, 20) -> 130
(20, 10, 30) -> 130
(20, 30, 10) -> 110
(30, 10, 20) -> 110
(30, 20, 10) -> 100
max khi cùng chiều: 140


### 3.4 `sorted(..., reverse=True)`, `zip` và modulo

- Sắp giảm dần: `sorted(x, reverse=True)`.
- `zip` ghép theo vị trí, dừng ở dãy ngắn hơn.
- Python không tràn số; lấy `% MOD` ở cuối là đủ (tổng tối đa ~`10⁵ · 10⁵ · 10⁵ = 10¹⁵`).

In [5]:
x, y = [5, 1, 3], [0, 2, 1]
print(sorted(x, reverse=True), sorted(y, reverse=True))
total = sum(p * q for p, q in zip(sorted(x, reverse=True), sorted(y, reverse=True)))
print("tổng =", total, "| % MOD =", total % MOD)
print("tổng lớn nhất có thể ~", 10**5 * 10**5 * 10**5)

[5, 3, 1] [2, 1, 0]
tổng = 13 | % MOD = 13
tổng lớn nhất có thể ~ 1000000000000000


### 3.5 Sắp xếp đếm (counting sort)

Khi giá trị nằm trong miền nhỏ `[0, K]`, đếm số lần xuất hiện mỗi giá trị rồi đọc ngược ⇒ sắp xếp trong `O(n + K)` không cần so sánh.

In [6]:
vals, K = [3, 0, 2, 3, 1, 0], 3
cnt = [0] * (K + 1)
for v in vals:
    cnt[v] += 1
desc = [v for v in range(K, -1, -1) for _ in range(cnt[v])]
print("đếm:", cnt, "| giảm dần:", desc)

đếm: [2, 1, 1, 2] | giảm dần: [3, 3, 2, 1, 0, 0]


## 4. Ý tưởng nền tảng / trực giác

Ví dụ 3: `nums = [1,2,3,4,5,10]`, `requests = [[0,2],[1,3],[1,1]]`.

**Đổi thứ tự cộng:** thay vì cộng theo từng truy vấn, ta đếm mỗi vị trí `i` được cộng bao nhiêu lần:

| vị trí | 0 | 1 | 2 | 3 | 4 | 5 |
|---|---|---|---|---|---|---|
| [0,2] | ✔ | ✔ | ✔ | | | |
| [1,3] | | ✔ | ✔ | ✔ | | |
| [1,1] | | ✔ | | | | |
| **freq** | 1 | 3 | 2 | 1 | 0 | 0 |

Tổng các truy vấn = `Σ nums_hoán_vị[i] · freq[i]`. Muốn lớn nhất ⇒ đặt số **lớn nhất** vào vị trí có `freq` **lớn nhất**:
- `freq` giảm dần: `[3, 2, 1, 1, 0, 0]`
- `nums` giảm dần: `[10, 5, 4, 3, 2, 1]`
- Tổng: `30 + 10 + 4 + 3 + 0 + 0 = 47`. ✔

Hoán vị cụ thể: vị trí 1 (freq 3) nhận 10, vị trí 2 (freq 2) nhận 5, vị trí 0 và 3 nhận 4 và 3 ⇒ `[4, 10, 5, 3, 2, 1]` — đúng như đề.

In [7]:
nums, req = [1, 2, 3, 4, 5, 10], [[0, 2], [1, 3], [1, 1]]
n = len(nums)
freq = [sum(1 for l, r in req if l <= i <= r) for i in range(n)]
print("freq =", freq)
pairs = list(zip(sorted(nums, reverse=True), sorted(freq, reverse=True)))
print("ghép (số, tần suất):", pairs, "-> tổng", sum(a * b for a, b in pairs))

freq = [1, 3, 2, 1, 0, 0]
ghép (số, tần suất): [(10, 3), (5, 2), (4, 1), (3, 1), (2, 0), (1, 0)] -> tổng 47


## 5. Cách 1 — Brute Force / baseline: thử mọi hoán vị

**Ý tưởng:** duyệt mọi hoán vị của `nums`; với mỗi hoán vị tính tổng các truy vấn bằng mảng tiền tố; lấy max.

In [8]:
def brute_force(nums: List[int], requests: List[List[int]]) -> int:
    best = 0
    for perm in set(permutations(nums)):          # set để bỏ hoán vị trùng khi nums có giá trị lặp
        pre = [0] + list(accumulate(perm))
        s = sum(pre[r + 1] - pre[l] for l, r in requests)
        best = max(best, s)
    return best % MOD

for nums, req, want in examples:
    print(brute_force(nums, req), "| mong đợi", want)

19 | mong đợi 19
11 | mong đợi 11
47 | mong đợi 47


**Độ phức tạp:** `O(n! · (n + q))` với `q = len(requests)`. `n = 8` đã là `40320` hoán vị; `n = 10⁵` là bất khả thi.

**Nút thắt:** ta đang thử **cách sắp xếp** — nhưng tổng chỉ phụ thuộc vào việc ghép mỗi giá trị với một **tần suất**. Khi biết `freq`, bài toán ghép cặp có lời giải tham lam trực tiếp.

In [9]:
rng = random.Random(1)
nums8 = [rng.randint(0, 9) for _ in range(8)]
req8 = [sorted([rng.randrange(8), rng.randrange(8)]) for _ in range(5)]
t0 = time.perf_counter()
r = brute_force(nums8, req8)
print(f"n=8: {r}, mất {time.perf_counter() - t0:.3f}s (n=12 sẽ chậm gấp ~12000 lần)")

n=8: 165, mất 0.006s (n=12 sẽ chậm gấp ~12000 lần)


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 — đổi thứ tự lấy tổng:**

`Σ_requests Σ_{i=l}^{r} A[i] = Σ_i A[i] · freq[i]`, với `freq[i] = #{truy vấn phủ i}`.

`freq` **không phụ thuộc** vào hoán vị — chỉ phụ thuộc vào `requests`.

**Insight 2 — bất đẳng thức hoán vị:** tổng `Σ A[i] · freq[i]` lớn nhất khi `A` và `freq` "cùng chiều" ⇒ sắp cả hai giảm dần (hoặc cùng tăng dần) rồi nhân theo vị trí.

**Insight 3 — tính `freq` nhanh:** lặp từng phần tử của mỗi truy vấn tốn `O(n · q)` (tới `10¹⁰`). Dùng **mảng hiệu**: mỗi truy vấn `O(1)`, cộng dồn `O(n)`.

**Quy tắc:** *nếu* một vị trí được nhiều truy vấn phủ hơn *thì* nó xứng đáng nhận số lớn hơn.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Đếm tần suất trực tiếp + sắp xếp ghép cặp

**Đoạn 1:** với mỗi truy vấn `[l, r]`, cộng `1` vào `freq[l..r]` bằng vòng lặp.

**Đoạn 2:** sắp `nums` và `freq` tăng dần, nhân theo vị trí, cộng dồn, lấy modulo.

Đúng nhưng đoạn 1 có thể tới `O(n · q)`.

In [10]:
def max_sum_naive_freq(nums: List[int], requests: List[List[int]]) -> int:
    n = len(nums)
    freq = [0] * n
    for l, r in requests:
        for i in range(l, r + 1):
            freq[i] += 1
    return sum(a * b for a, b in zip(sorted(nums), sorted(freq))) % MOD

for nums, req, want in examples:
    print(max_sum_naive_freq(nums, req), "| mong đợi", want)

19 | mong đợi 19
11 | mong đợi 11
47 | mong đợi 47


Code nộp LeetCode cho cách 2 (có thể **TLE** với dữ liệu lớn):

```python
class Solution:
    def maxSumRangeQuery(self, nums: List[int], requests: List[List[int]]) -> int:
        n = len(nums)
        freq = [0] * n
        for l, r in requests:
            for i in range(l, r + 1):
                freq[i] += 1
        return sum(a * b for a, b in zip(sorted(nums), sorted(freq))) % (10**9 + 7)
```

**Độ phức tạp:** `O(n·q + n log n)` thời gian, `O(n)` bộ nhớ.

### 7.2 Cách 3 — Mảng hiệu + sắp xếp (lời giải chuẩn)

**Đoạn 1:** `d = [0] * (n + 1)`; với mỗi `[l, r]`: `d[l] += 1`, `d[r + 1] -= 1`.

**Đoạn 2:** cộng dồn `d[0..n-1]` ⇒ `freq`.

**Đoạn 3:** sắp `nums` và `freq`, nhân theo vị trí, modulo.

In [11]:
def max_sum_diff(nums: List[int], requests: List[List[int]]) -> int:
    n = len(nums)
    d = [0] * (n + 1)
    for l, r in requests:
        d[l] += 1
        d[r + 1] -= 1
    freq = list(accumulate(d[:n]))
    nums_sorted = sorted(nums)
    freq.sort()
    return sum(a * b for a, b in zip(nums_sorted, freq)) % MOD

for nums, req, want in examples:
    print(max_sum_diff(nums, req), "| mong đợi", want)

19 | mong đợi 19
11 | mong đợi 11
47 | mong đợi 47


Code nộp LeetCode cho cách 3 (giống phần 9):

```python
class Solution:
    def maxSumRangeQuery(self, nums: List[int], requests: List[List[int]]) -> int:
        n = len(nums)
        d = [0] * (n + 1)
        for l, r in requests:
            d[l] += 1
            d[r + 1] -= 1
        freq = list(accumulate(d[:n]))
        nums.sort()
        freq.sort()
        return sum(a * b for a, b in zip(nums, freq)) % (10**9 + 7)
```

**Độ phức tạp:** `O(n log n + q)` thời gian, `O(n)` bộ nhớ.

### 7.3 Cách 4 — Mảng hiệu + sắp xếp đếm (`O(n + q + K)`)

Cả `nums[i] ≤ 10⁵` và `freq[i] ≤ q ≤ 10⁵` đều là số nguyên nhỏ ⇒ thay `sort` bằng **đếm**:
- `cnt_val[v]` = số phần tử `nums` bằng `v`; `cnt_freq[f]` = số vị trí có tần suất `f`.
- Hai con trỏ đi từ giá trị lớn nhất và tần suất lớn nhất, mỗi lần ghép `k = min(cnt_val[v], cnt_freq[f])` cặp, cộng `k · v · f`.

Đây là ghép "lớn với lớn" theo từng **khối giá trị**, không cần sắp từng phần tử.

In [12]:
def max_sum_counting(nums: List[int], requests: List[List[int]]) -> int:
    n = len(nums)
    d = [0] * (n + 1)
    for l, r in requests:
        d[l] += 1
        d[r + 1] -= 1
    cnt_freq = [0] * (len(requests) + 1)
    cur = 0
    for i in range(n):
        cur += d[i]
        cnt_freq[cur] += 1
    cnt_val = [0] * (max(nums) + 1)
    for v in nums:
        cnt_val[v] += 1
    total = 0
    v, f = len(cnt_val) - 1, len(cnt_freq) - 1
    while v >= 0 and f >= 0:
        if cnt_val[v] == 0:
            v -= 1
            continue
        if cnt_freq[f] == 0:
            f -= 1
            continue
        k = min(cnt_val[v], cnt_freq[f])
        total += k * v * f
        cnt_val[v] -= k
        cnt_freq[f] -= k
    return total % MOD

for nums, req, want in examples:
    print(max_sum_counting(nums, req), "| mong đợi", want)

19 | mong đợi 19
11 | mong đợi 11
47 | mong đợi 47


Code nộp LeetCode cho cách 4:

```python
class Solution:
    def maxSumRangeQuery(self, nums: List[int], requests: List[List[int]]) -> int:
        n = len(nums)
        d = [0] * (n + 1)
        for l, r in requests:
            d[l] += 1
            d[r + 1] -= 1
        cnt_freq = [0] * (len(requests) + 1)
        cur = 0
        for i in range(n):
            cur += d[i]
            cnt_freq[cur] += 1
        cnt_val = [0] * (max(nums) + 1)
        for v in nums:
            cnt_val[v] += 1
        total = 0
        v, f = len(cnt_val) - 1, len(cnt_freq) - 1
        while v >= 0 and f >= 0:
            if cnt_val[v] == 0:
                v -= 1
            elif cnt_freq[f] == 0:
                f -= 1
            else:
                k = min(cnt_val[v], cnt_freq[f])
                total += k * v * f
                cnt_val[v] -= k
                cnt_freq[f] -= k
        return total % (10**9 + 7)
```

**Độ phức tạp:** `O(n + q + max(nums))` thời gian, `O(n + q + max(nums))` bộ nhớ. Nhanh hơn về lý thuyết; trong Python, `sort` (viết bằng C) thường đã rất nhanh nên cách 3 vẫn là lựa chọn mặc định.

## 8. TRACE từng bước

Cách 3 với ví dụ 3: `nums = [1,2,3,4,5,10]`, `requests = [[0,2],[1,3],[1,1]]`, `n = 6`.

| bước | thao tác | d (độ dài 7) |
|---|---|---|
| khởi tạo | | `[0,0,0,0,0,0,0]` |
| [0,2] | d[0]+=1, d[3]-=1 | `[1,0,0,-1,0,0,0]` |
| [1,3] | d[1]+=1, d[4]-=1 | `[1,1,0,-1,-1,0,0]` |
| [1,1] | d[1]+=1, d[2]-=1 | `[1,2,-1,-1,-1,0,0]` |
| cộng dồn | `freq = [1,3,2,1,0,0]` | |

Sắp tăng: `nums → [1,2,3,4,5,10]`, `freq → [0,0,1,1,2,3]`. Tích theo vị trí: `0,0,3,4,10,30` ⇒ tổng **47**.

In [13]:
def trace_diff(nums, requests):
    n = len(nums)
    d = [0] * (n + 1)
    print("khởi tạo d =", d)
    for l, r in requests:
        d[l] += 1
        d[r + 1] -= 1
        print(f"truy vấn [{l},{r}]: d[{l}] += 1, d[{r + 1}] -= 1 -> d = {d}")
    freq, cur = [], 0
    for i in range(n):
        cur += d[i]
        freq.append(cur)
        print(f"  cộng dồn i={i}: cur={cur}")
    print("freq =", freq)
    a, b = sorted(nums), sorted(freq)
    total = 0
    for x, f in zip(a, b):
        total += x * f
        print(f"  ghép số {x:>2} với tần suất {f}: +{x * f:>2} -> {total}")
    return total % MOD

print("kết quả:", trace_diff([1, 2, 3, 4, 5, 10], [[0, 2], [1, 3], [1, 1]]))

khởi tạo d = [0, 0, 0, 0, 0, 0, 0]
truy vấn [0,2]: d[0] += 1, d[3] -= 1 -> d = [1, 0, 0, -1, 0, 0, 0]
truy vấn [1,3]: d[1] += 1, d[4] -= 1 -> d = [1, 1, 0, -1, -1, 0, 0]
truy vấn [1,1]: d[1] += 1, d[2] -= 1 -> d = [1, 2, -1, -1, -1, 0, 0]
  cộng dồn i=0: cur=1
  cộng dồn i=1: cur=3
  cộng dồn i=2: cur=2
  cộng dồn i=3: cur=1
  cộng dồn i=4: cur=0
  cộng dồn i=5: cur=0
freq = [1, 3, 2, 1, 0, 0]
  ghép số  1 với tần suất 0: + 0 -> 0
  ghép số  2 với tần suất 0: + 0 -> 0
  ghép số  3 với tần suất 1: + 3 -> 3
  ghép số  4 với tần suất 1: + 4 -> 7
  ghép số  5 với tần suất 2: +10 -> 17
  ghép số 10 với tần suất 3: +30 -> 47
kết quả: 47


In [14]:
def build_permutation(nums, requests):
    """Dựng một hoán vị tối ưu: vị trí có tần suất cao nhận số lớn nhất."""
    n = len(nums)
    d = [0] * (n + 1)
    for l, r in requests:
        d[l] += 1
        d[r + 1] -= 1
    freq = list(accumulate(d[:n]))
    order = sorted(range(n), key=lambda i: -freq[i])      # vị trí theo tần suất giảm dần
    perm = [0] * n
    for pos, val in zip(order, sorted(nums, reverse=True)):
        perm[pos] = val
    return perm

perm = build_permutation([1, 2, 3, 4, 5, 10], [[0, 2], [1, 3], [1, 1]])
pre = [0] + list(accumulate(perm))
print("hoán vị tối ưu:", perm, "| tổng từng truy vấn:", [pre[r + 1] - pre[l] for l, r in [[0, 2], [1, 3], [1, 1]]])

hoán vị tối ưu: [4, 10, 5, 3, 2, 1] | tổng từng truy vấn: [19, 18, 10]


## 9. Code nộp LeetCode

In [15]:
class Solution:
    def maxSumRangeQuery(self, nums: List[int], requests: List[List[int]]) -> int:
        n = len(nums)
        d = [0] * (n + 1)
        for l, r in requests:
            d[l] += 1
            d[r + 1] -= 1
        freq = list(accumulate(d[:n]))
        nums.sort()
        freq.sort()
        return sum(a * b for a, b in zip(nums, freq)) % (10**9 + 7)

def solution_copy(nums, requests):
    return Solution().maxSumRangeQuery(list(nums), [list(r) for r in requests])   # tránh sửa dữ liệu test

for nums, req, want in examples:
    print(solution_copy(nums, req), "| mong đợi", want)

19 | mong đợi 19
11 | mong đợi 11
47 | mong đợi 47


## 10. Vì sao thuật toán đúng

**(1) Đổi thứ tự tổng.** Với một hoán vị `A`, tổng các truy vấn là `Σ_{(l,r)} Σ_{i=l}^{r} A[i]`. Mỗi `A[i]` xuất hiện đúng `freq[i]` lần (một lần cho mỗi truy vấn phủ `i`) ⇒ tổng `= Σ_i A[i] · freq[i]`.

**(2) Mảng hiệu cho đúng `freq`.** Sau khi cộng dồn, `Σ_{k ≤ i} d[k]` = (số truy vấn có `l ≤ i`) − (số truy vấn có `r + 1 ≤ i`) = số truy vấn có `l ≤ i ≤ r` = `freq[i]`.

**(3) Ghép cùng chiều là tối ưu (lập luận trao đổi).** Giả sử có hai vị trí `i, j` với `freq[i] > freq[j]` nhưng `A[i] < A[j]`. Hoán đổi `A[i]` và `A[j]` làm tổng thay đổi một lượng

`(A[j]·freq[i] + A[i]·freq[j]) − (A[i]·freq[i] + A[j]·freq[j]) = (A[j] − A[i])(freq[i] − freq[j]) > 0`.

Vậy mọi hoán vị có "cặp ngược chiều" đều cải thiện được ⇒ tối ưu đạt khi `A` và `freq` cùng thứ tự, tức ghép sau khi sắp cả hai.

**(4) Cách 4** ghép đúng như sắp giảm dần cả hai dãy, chỉ là gộp các phần tử bằng nhau thành khối ⇒ cùng kết quả.

## 11. Phân tích Time / Space

Gọi `n = len(nums)`, `q = len(requests)`, `K = max(nums)`.

| Cách | Time | Space phụ |
|---|---|---|
| 1. Thử mọi hoán vị | `O(n! · (n + q))` | `O(n)` |
| 2. Đếm `freq` trực tiếp + sắp | `O(n·q + n log n)` | `O(n)` |
| 3. Mảng hiệu + sắp | `O(q + n log n)` | `O(n)` |
| 4. Mảng hiệu + sắp xếp đếm | `O(n + q + K)` | `O(n + q + K)` |

Cách 2 xấu nhất khi mọi truy vấn là `[0, n-1]`: `10⁵ × 10⁵ = 10¹⁰` bước.

## 12. Bộ test

In [16]:
ALL = [brute_force, max_sum_naive_freq, max_sum_diff, max_sum_counting, solution_copy]

for f in ALL:
    for nums, req, want in examples:
        assert f(list(nums), [list(r) for r in req]) == want, (f.__name__, nums, req)
print("Ví dụ đề: OK")

edge = [
    ([7], [[0, 0]], 7),                        # n = 1
    ([0, 0, 0], [[0, 2], [1, 1]], 0),          # toàn 0
    ([5, 5, 5], [[0, 0], [2, 2]], 10),         # giá trị bằng nhau
    ([1, 2, 3], [[0, 2], [0, 2], [0, 2]], 18),  # mọi truy vấn phủ hết: 3 * (1+2+3)
    ([9, 1, 1, 1], [[3, 3], [3, 3]], 18),       # cùng một vị trí được hỏi 2 lần
]
for f in ALL:
    for nums, req, want in edge:
        assert f(list(nums), [list(r) for r in req]) == want, (f.__name__, nums, req, f(list(nums), req))
print("Edge case: OK")

Ví dụ đề: OK
Edge case: OK


In [17]:
# Ngẫu nhiên seed cố định: đối chiếu với vét cạn hoán vị (n ≤ 7)
rng = random.Random(1589)
for _ in range(600):
    n = rng.randint(1, 7)
    nums = [rng.randint(0, rng.choice([3, 20, 10**5])) for _ in range(n)]
    req = []
    for _ in range(rng.randint(1, 6)):
        l = rng.randrange(n)
        req.append([l, rng.randint(l, n - 1)])
    want = brute_force(nums, req)
    for f in ALL[1:]:
        assert f(list(nums), [list(r) for r in req]) == want, (f.__name__, nums, req)
print("600 bộ ngẫu nhiên khớp vét cạn: OK")

# Hoán vị dựng được thật sự đạt tổng tối ưu
for _ in range(300):
    n = rng.randint(1, 30)
    nums = [rng.randint(0, 50) for _ in range(n)]
    req = []
    for _ in range(rng.randint(1, 20)):
        l = rng.randrange(n)
        req.append([l, rng.randint(l, n - 1)])
    perm = build_permutation(nums, req)
    pre = [0] + list(accumulate(perm))
    assert sorted(perm) == sorted(nums)
    assert sum(pre[r + 1] - pre[l] for l, r in req) % MOD == max_sum_diff(nums, req)
print("Hoán vị dựng được đạt tối ưu: OK")

600 bộ ngẫu nhiên khớp vét cạn: OK
Hoán vị dựng được đạt tối ưu: OK


In [18]:
# Test lớn đúng ràng buộc + kiểm tra modulo
rng = random.Random(9)
n, q = 10**5, 10**5
nums_big = [rng.randint(0, 10**5) for _ in range(n)]
req_big = []
for _ in range(q):
    l = rng.randrange(n)
    req_big.append([l, rng.randint(l, n - 1)])
res = []
for f in [max_sum_diff, max_sum_counting, solution_copy]:
    t0 = time.perf_counter()
    res.append(f(nums_big, req_big))
    print(f"  {f.__name__:<16} {res[-1]:>10}  {time.perf_counter() - t0:.3f}s")
assert len(set(res)) == 1

# Trường hợp lớn nhất: mọi truy vấn [0, n-1], mọi số = 10^5 -> tổng thô 10^15 phải modulo
worst_nums, worst_req = [10**5] * n, [[0, n - 1]] * q
raw = 10**5 * n * q
assert max_sum_diff(worst_nums, worst_req) == raw % MOD
print("tổng thô =", raw, "| sau modulo =", raw % MOD)

  max_sum_diff       87053930  0.045s


  max_sum_counting   87053930  0.051s
  solution_copy      87053930  0.061s
tổng thô = 1000000000000000 | sau modulo = 993000007


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Sắp **ngược chiều** (số lớn ghép với tần suất nhỏ)

Bất đẳng thức hoán vị cho biết ghép ngược chiều chính là cho **tổng nhỏ nhất**.

In [19]:
def wrong_opposite(nums, requests):
    n = len(nums)
    d = [0] * (n + 1)
    for l, r in requests:
        d[l] += 1
        d[r + 1] -= 1
    freq = list(accumulate(d[:n]))
    return sum(a * b for a, b in zip(sorted(nums), sorted(freq, reverse=True))) % MOD   # SAI

nums, req, want = examples[2]
print("sai:", wrong_opposite(nums, req), "| đúng:", solution_copy(nums, req))

sai: 14 | đúng: 47


### Lỗi 2 — Mảng hiệu chỉ có `n` phần tử

Khi `end = n − 1`, `d[end + 1] = d[n]` vượt chỉ số ⇒ `IndexError`. Ta bắt lỗi để in ra.

In [20]:
def wrong_diff_size(nums, requests):
    n = len(nums)
    d = [0] * n                                    # SAI: phải là n + 1
    for l, r in requests:
        d[l] += 1
        d[r + 1] -= 1
    freq = list(accumulate(d))
    return sum(a * b for a, b in zip(sorted(nums), sorted(freq))) % MOD

nums, req, want = examples[1]
try:
    print(wrong_diff_size(nums, [[0, 5]]))
except IndexError as e:
    print("IndexError:", e, "| đúng:", solution_copy(nums, [[0, 5]]))

IndexError: list index out of range | đúng: 21


### Lỗi 3 — Đặt `−1` tại `end` thay vì `end + 1`

Đoạn `[l, r]` gồm cả `r`, nên dấu trừ phải đặt **sau** `r`. Đặt tại `r` làm vị trí `r` mất một lần đếm.

In [21]:
def wrong_minus_at_end(nums, requests):
    n = len(nums)
    d = [0] * (n + 1)
    for l, r in requests:
        d[l] += 1
        d[r] -= 1                                  # SAI: phải là d[r + 1]
    freq = list(accumulate(d[:n]))
    return sum(a * b for a, b in zip(sorted(nums), sorted(freq))) % MOD

nums, req, want = examples[0]
print("sai:", wrong_minus_at_end(nums, req), "| đúng:", solution_copy(nums, req))

sai: 12 | đúng: 19


### Lỗi 4 — Quên modulo

Python trả về số rất lớn (tới `10¹⁵`) — LeetCode báo **Wrong Answer**.

In [22]:
def wrong_no_mod(nums, requests):
    n = len(nums)
    d = [0] * (n + 1)
    for l, r in requests:
        d[l] += 1
        d[r + 1] -= 1
    freq = list(accumulate(d[:n]))
    return sum(a * b for a, b in zip(sorted(nums), sorted(freq)))     # SAI: thiếu % MOD

nums_w, req_w = [10**5] * 1000, [[0, 999]] * 20000
print("sai:", wrong_no_mod(nums_w, req_w), "| đúng:", solution_copy(nums_w, req_w))

sai: 2000000000000 | đúng: 999986007


### Lỗi 5 — Tham lam theo từng truy vấn: lấp số lớn vào truy vấn đầu tiên trước

Xử lý truy vấn theo thứ tự đầu vào, đặt các số lớn nhất còn lại vào các vị trí trống của truy vấn hiện tại — bỏ qua việc một vị trí có thể được **nhiều** truy vấn chia sẻ.

In [23]:
def wrong_greedy_by_request(nums, requests):
    n = len(nums)
    pool = sorted(nums)
    perm = [None] * n
    for l, r in requests:
        for i in range(l, r + 1):
            if perm[i] is None:
                perm[i] = pool.pop()                 # số lớn nhất còn lại
    for i in range(n):
        if perm[i] is None:
            perm[i] = pool.pop()
    pre = [0] + list(accumulate(perm))
    return sum(pre[r + 1] - pre[l] for l, r in requests) % MOD   # SAI: không tối ưu

nums, req, want = examples[2]
print("sai:", wrong_greedy_by_request(nums, req), "| đúng:", solution_copy(nums, req))

sai: 36 | đúng: 47


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Tổng của nhiều truy vấn đoạn" + "được sắp xếp lại"** ⇒ đếm **tần suất** mỗi vị trí, rồi **ghép lớn với lớn**.
- **Nhiều cập nhật `+1` trên đoạn, chỉ cần kết quả cuối** ⇒ **mảng hiệu** (`d[l] += 1`, `d[r+1] -= 1`, cộng dồn).
- Ghép hai dãy để tối đa/tối thiểu tổng tích ⇒ **bất đẳng thức hoán vị**: cùng chiều (max), ngược chiều (min).
- Nhớ kích thước `n + 1` cho mảng hiệu, và `% (10**9 + 7)` ở cuối.
- Kiểm tra nhanh: một truy vấn `[0, n−1]` ⇒ đáp án = tổng `nums`; mọi truy vấn trùng nhau ⇒ nhân số truy vấn.

## 15. Biến thể của bài

1. **Tổng nhỏ nhất** trên mọi hoán vị ⇒ ghép ngược chiều (bài tập 4).
2. **Truy vấn có trọng số** `[l, r, w]` (được tính `w` lần), `nums` có thể âm ⇒ mảng hiệu cộng `w`; vẫn ghép cùng chiều (bài tập 5).
3. **LC 370 — Range Addition / LC 1109 — Corporate Flight Bookings:** mảng hiệu thuần tuý.
4. **LC 1094 — Car Pooling:** mảng hiệu để kiểm tra sức chứa tại mọi thời điểm.
5. **Mảng hiệu 2D** (LC 2536 — Increment Submatrices by One): `+1` ở góc trên trái, `−1` ở hai góc, `+1` ở góc dưới phải.
6. **LC 1874 — Minimize Product Sum of Two Arrays:** áp dụng trực tiếp bất đẳng thức hoán vị.

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử mọi hoán vị | vét cạn | `O(n!·(n+q))` | `O(n)` | kiểm thử, `n ≤ 8` |
| 2. Đếm trực tiếp | lặp từng phần tử của mỗi truy vấn | `O(n·q + n log n)` | `O(n)` | dữ liệu nhỏ, để hiểu ý tưởng |
| 3. Mảng hiệu + sắp | `freq` bằng mảng hiệu, ghép lớn–lớn | `O(q + n log n)` | `O(n)` | **lời giải chuẩn** |
| 4. Mảng hiệu + đếm | ghép theo khối giá trị, không sắp | `O(n + q + K)` | `O(n + q + K)` | giá trị nhỏ, cần tuyến tính |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `maxSumRangeQuery([5, 1, 3], [[0, 2], [1, 1]])` trả về bao nhiêu?

**Bài 2 (tính tay):** Với `n = 5` và `requests = [[0,4],[1,2],[2,2]]`, viết mảng hiệu `d` (độ dài 6) sau mọi truy vấn và mảng `freq`.

**Bài 3 (điền TODO):** Viết `ex3_optimal_permutation(nums, requests)` trả về **một hoán vị** của `nums` đạt tổng lớn nhất.

**Bài 4 (tự viết lại):** Viết `ex4_min_sum(nums, requests)` trả về tổng **nhỏ nhất** trên mọi hoán vị (không cần modulo).

**Bài 5 (mini challenge):** Truy vấn có dạng `[l, r, w]` (`w ≥ 1`, truy vấn được tính `w` lần) và `nums` có thể **âm**. Viết `ex5_max_weighted(nums, requests)` trả về tổng lớn nhất (không modulo).

In [24]:
# Bài 3 — khung TODO (chỉ định nghĩa, KHÔNG gọi)
def ex3_optimal_permutation(nums, requests):
    n = len(nums)
    perm = [0] * n
    # TODO: tính freq bằng mảng hiệu
    # TODO: sắp các vị trí theo freq giảm dần
    # TODO: gán lần lượt các số của nums (giảm dần) vào các vị trí đó
    return perm

In [25]:
# Bài 5 — khung TODO (chỉ định nghĩa, KHÔNG gọi)
def ex5_max_weighted(nums, requests):
    n = len(nums)
    d = [0] * (n + 1)
    # TODO: với mỗi [l, r, w]: d[l] += w, d[r + 1] -= w
    # TODO: cộng dồn ra freq; sắp nums và freq cùng chiều; trả tổng tích
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `freq = [1, 2, 1]`. (2) Số lớn nhất `5` vào vị trí có tần suất 2. (3) `5·2 + 3·1 + 1·1`.

**Bài 2.** (1) `[0,4]`: `d[0]+=1, d[5]-=1`. (2) `[1,2]`: `d[1]+=1, d[3]-=1`; `[2,2]`: `d[2]+=1, d[3]-=1`. (3) Cộng dồn.

**Bài 3.** (1) Dùng `sorted(range(n), key=lambda i: -freq[i])`. (2) Ghép với `sorted(nums, reverse=True)`. (3) Kiểm tra: `sorted(perm) == sorted(nums)` và tổng truy vấn bằng đáp án.

**Bài 4.** (1) Bất đẳng thức hoán vị: ngược chiều cho min. (2) Sắp `nums` tăng, `freq` giảm. (3) So với vét cạn hoán vị.

**Bài 5.** (1) Mảng hiệu cộng `w` thay vì `1`. (2) Bất đẳng thức hoán vị đúng cho mọi số thực, kể cả âm. (3) Vị trí có `freq = 0` vẫn phải nhận một số (có thể âm) — đóng góp 0, không ảnh hưởng.

## 19. Đáp án tham khảo cho bài tập

In [26]:
# Bài 1
ans1 = solution_copy([5, 1, 3], [[0, 2], [1, 1]])
print("Bài 1:", ans1)
assert ans1 == 5 * 2 + 3 + 1 == brute_force([5, 1, 3], [[0, 2], [1, 1]])

# Bài 2
n2, req2 = 5, [[0, 4], [1, 2], [2, 2]]
d2 = [0] * (n2 + 1)
for l, r in req2:
    d2[l] += 1
    d2[r + 1] -= 1
freq2 = list(accumulate(d2[:n2]))
print("Bài 2: d =", d2, "| freq =", freq2)
assert d2 == [1, 1, 1, -2, 0, -1] and freq2 == [1, 2, 3, 1, 1]
print("Bài 1, 2: OK")

Bài 1: 14
Bài 2: d = [1, 1, 1, -2, 0, -1] | freq = [1, 2, 3, 1, 1]
Bài 1, 2: OK


In [27]:
# Bài 3
def ex3_optimal_permutation_sol(nums, requests):
    n = len(nums)
    d = [0] * (n + 1)
    for l, r in requests:
        d[l] += 1
        d[r + 1] -= 1
    freq = list(accumulate(d[:n]))
    perm = [0] * n
    for pos, val in zip(sorted(range(n), key=lambda i: -freq[i]), sorted(nums, reverse=True)):
        perm[pos] = val
    return perm

print("Bài 3:", ex3_optimal_permutation_sol([1, 2, 3, 4, 5], [[1, 3], [0, 1]]))
rng = random.Random(3)
for _ in range(500):
    n = rng.randint(1, 7)
    nums = [rng.randint(0, 9) for _ in range(n)]
    req = []
    for _ in range(rng.randint(1, 5)):
        l = rng.randrange(n)
        req.append([l, rng.randint(l, n - 1)])
    perm = ex3_optimal_permutation_sol(nums, req)
    pre = [0] + list(accumulate(perm))
    assert sorted(perm) == sorted(nums)
    assert sum(pre[r + 1] - pre[l] for l, r in req) % MOD == brute_force(nums, req)
print("Bài 3: OK")

Bài 3: [4, 5, 3, 2, 1]


Bài 3: OK


In [28]:
# Bài 4
def ex4_min_sum(nums, requests):
    n = len(nums)
    d = [0] * (n + 1)
    for l, r in requests:
        d[l] += 1
        d[r + 1] -= 1
    freq = list(accumulate(d[:n]))
    return sum(a * b for a, b in zip(sorted(nums), sorted(freq, reverse=True)))

def ex4_brute(nums, requests):
    best = None
    for perm in set(permutations(nums)):
        pre = [0] + list(accumulate(perm))
        s = sum(pre[r + 1] - pre[l] for l, r in requests)
        best = s if best is None else min(best, s)
    return best

print("Bài 4:", ex4_min_sum([1, 2, 3, 4, 5, 10], [[0, 2], [1, 3], [1, 1]]))
rng = random.Random(4)
for _ in range(500):
    n = rng.randint(1, 7)
    nums = [rng.randint(0, 9) for _ in range(n)]
    req = []
    for _ in range(rng.randint(1, 5)):
        l = rng.randrange(n)
        req.append([l, rng.randint(l, n - 1)])
    assert ex4_min_sum(nums, req) == ex4_brute(nums, req)
print("Bài 4: OK")

Bài 4: 14


Bài 4: OK


In [29]:
# Bài 5
def ex5_max_weighted_sol(nums, requests):
    n = len(nums)
    d = [0] * (n + 1)
    for l, r, w in requests:
        d[l] += w
        d[r + 1] -= w
    freq = list(accumulate(d[:n]))
    return sum(a * b for a, b in zip(sorted(nums), sorted(freq)))

def ex5_brute(nums, requests):
    best = None
    for perm in set(permutations(nums)):
        pre = [0] + list(accumulate(perm))
        s = sum(w * (pre[r + 1] - pre[l]) for l, r, w in requests)
        best = s if best is None else max(best, s)
    return best

print("Bài 5:", ex5_max_weighted_sol([-3, 4, 1], [[0, 1, 2], [2, 2, 5]]))
assert ex5_max_weighted_sol([-3, 4, 1], [[0, 1, 2], [2, 2, 5]]) == ex5_brute([-3, 4, 1], [[0, 1, 2], [2, 2, 5]])
rng = random.Random(5)
for _ in range(500):
    n = rng.randint(1, 6)
    nums = [rng.randint(-9, 9) for _ in range(n)]
    req = []
    for _ in range(rng.randint(1, 5)):
        l = rng.randrange(n)
        req.append([l, rng.randint(l, n - 1), rng.randint(1, 4)])
    assert ex5_max_weighted_sol(nums, req) == ex5_brute(nums, req), (nums, req)
print("Bài 5: OK")

Bài 5: 16
Bài 5: OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
class Solution:
    def maxSumRangeQuery(self, nums, requests):
        n = len(nums)
        d = [0] * (n + 1)                      # mảng hiệu, độ dài n + 1
        for l, r in requests:
            d[l] += 1
            d[r + 1] -= 1
        freq = list(accumulate(d[:n]))         # số truy vấn phủ mỗi vị trí
        nums.sort()
        freq.sort()                            # ghép cùng chiều: lớn với lớn
        return sum(a * b for a, b in zip(nums, freq)) % (10**9 + 7)
```

**Template "mảng hiệu":** `d[l] += v; d[r+1] -= v` cho mỗi đoạn → cộng dồn tiền tố → giá trị tại từng vị trí.

**Checklist trước khi submit:**
- [ ] Mảng hiệu độ dài `n + 1`, trừ tại `r + 1`.
- [ ] Sắp `nums` và `freq` **cùng chiều**.
- [ ] Lấy `% (10**9 + 7)` ở kết quả.
- [ ] Không cần tạo hoán vị thật — chỉ cần tổng.
- [ ] Thử nhanh: ví dụ → `19, 11, 47`; `[7]`,`[[0,0]]` → `7`; toàn 0 → `0`.